In [ ]:
import warnings
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix

warnings.filterwarnings('ignore')

sc.settings.verbosity = 0
sc.settings.set_figure_params(dpi=150, facecolor='white', frameon=False)
plt.rcParams['axes.grid'] = False

plot_directory = "/home/ali/Desktop/scRNAseq/2024_Ali_scRNAseq/HumanLungCellAtlas/2603_HLCA_Stroma"
os.makedirs(plot_directory, exist_ok=True)

In [ ]:
file_path = "/home/ali/Desktop/scRNAseq/2024_Ali_scRNAseq/HumanLungCellAtlas/Objects/HCLA_full_scanpy.h5ad"
adata = sc.read_h5ad(file_path)
adata

In [ ]:
unique_names = adata.obs['ann_level_1'].unique()
print(unique_names)

In [ ]:
adata.obs['disease'] = adata.obs['disease'].astype('category')
print(adata.obs['disease'].cat.categories.tolist())

In [ ]:
sc.pl.umap(adata, color='ann_level_1', palette='Set3', frameon=False)

In [ ]:
sc.pl.umap(adata, color=['SCUBE2', 'ADAMTS4', 'CTHRC1'], gene_symbols='feature_name', size=5, ncols=3, frameon=False, color_map='RdYlBu_r')

In [ ]:
# Define target subsets
target_diseases = ['pulmonary fibrosis', 'interstitial lung disease', 'non-specific interstitial pneumonia', 'normal']
target_cells = ['Stroma']

# Filter and clean
adata = adata[adata.obs['disease'].isin(target_diseases)].copy()
adata = adata[adata.obs['ann_level_1'].isin(target_cells)].copy()

# Fix indexing and categories
adata.var_names_make_unique()
if adata.obs['disease'].dtype.name == 'category':
    adata.obs['disease'] = adata.obs['disease'].cat.remove_unused_categories()
if adata.obs['ann_level_1'].dtype.name == 'category':
    adata.obs['ann_level_1'] = adata.obs['ann_level_1'].cat.remove_unused_categories()

print(f"Remaining: {adata.n_obs} cells, {adata.n_vars} genes")
print(f"Diseases: {adata.obs['disease'].unique().tolist()}")

In [ ]:
adata

In [ ]:
sc.pl.umap(adata, color='ann_level_1')

In [ ]:
### Outlier removal and UMAP cleanup 
u = adata.obsm['X_umap']
m, s = u.mean(0), u.std(0)

low, high = m - 3 * s, m + 3 * s
mask = np.all((u >= low) & (u <= high), axis=1)

adata = adata[mask].copy()
adata.var_names = adata.var_names.astype(str)

sc.pl.umap(adata, color='ann_level_1', size=20)

In [ ]:
sc.pl.umap(adata, color='study', size=5)

In [ ]:
from scipy.sparse import issparse

genes = ['CTHRC1', 'FGF10',  'SCUBE2', 'INMT', 'ADAMTS4', 'FGF18', 'PDGFRA', 'PDGFRB', 'SFRP1']
disease_categories = adata.obs['disease'].unique()
ncols = 4
nrows = 1
cmap = 'gist_heat_r'

for gene in genes:
    gene_idx = np.where(adata.var['feature_name'] == gene)[0]
    if len(gene_idx) == 0:
        print(f"Gene {gene} not found in feature_name column")
        continue
    gene_idx = gene_idx[0]

    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(5 * ncols, 5 * nrows))
    if ncols == 1:  # ensure axes is iterable
        axes = [axes]
    axes = axes.flatten()

    adata.obs['Temp_Gene_Expression'] = np.nan

    for i, disease in enumerate(disease_categories):
        disease_mask = adata.obs['disease'] == disease
        gene_expression = adata[:, gene_idx].X.toarray().flatten() if issparse(adata[:, gene_idx].X) else adata[:, gene_idx].X.flatten()
        adata.obs['Temp_Gene_Expression'] = np.where(disease_mask, gene_expression, np.nan)

        sc.pl.umap(
            adata,
            color='Temp_Gene_Expression',
            color_map=cmap,
            size=45,
            title=f"{disease}",
            ax=axes[i],
            show=False,
            na_color='#d3d3d3',
            alpha=0.8
        )
        axes[i].tick_params(axis='both', which='both', length=0)

    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.suptitle(f"{gene}", fontsize=18, weight='bold')
    plt.tight_layout()
    save_file = f"UMAP_{gene}_by_disease.png"
    plt.savefig(os.path.join(plot_directory, save_file), dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved {save_file}")

In [ ]:
#### Save this ILD/IPF adata for later use 
adata.write_h5ad("/home/ali/Desktop/scRNAseq/2024_Ali_scRNAseq/HumanLungCellAtlas/Objects/2603_ILDIPFNSIP_StromaOnly.h5ad")